# 07. 미니 프로젝트 — 검색 + 요약 에이전트

입문 과정에서 배운 내용을 조합하여, Tavily 웹 검색 도구를 갖춘 리서치 에이전트를 만듭니다.

## 학습 목표

- Tavily 검색 도구를 직접 정의한다
- Deep Agents로 리서치 에이전트를 만든다
- 스트리밍으로 에이전트 실행 과정을 실시간 관찰한다
- LangChain 에이전트로도 같은 작업을 수행하여 비교한다

In [1]:
import sys
!uv pip install --python {sys.executable} "tavily-python>=0.5.0"


Using Python 3.12.8 environment at: C:\Users\Soonju\Downloads\POSCODX\POSCODX\soonju\posco-dx-agent-dev-lang-day2\.venv
Checked 1 package in 167ms


## 7.1 환경 설정

이 노트북에는 `TAVILY_API_KEY`가 필요합니다. https://tavily.com 에서 무료로 발급받을 수 있습니다.

In [1]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY 필요!"
assert os.environ.get("TAVILY_API_KEY"), "TAVILY_API_KEY 필요!"

from langchain_openai import ChatOpenAI
model = ChatOpenAI(model="gpt-5.4")
print("\u2713 환경 준비 완료")

✓ 환경 준비 완료


In [2]:
# Observability 설정 (선택) - LangSmith 또는 Langfuse
# .env에 키를 설정하거나, 아래 주석을 해제하여 직접 입력하세요.
# os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."
# os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."
# os.environ["LANGFUSE_HOST"] = "https://lf.ddok.ai"
import os

# LangSmith: LANGSMITH_TRACING=true 시 자동 활성화 (코드 수정 불필요)
if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_API_KEY", os.environ.get("LANGSMITH_API_KEY", ""))
    os.environ.setdefault("LANGCHAIN_PROJECT", os.environ.get("LANGSMITH_PROJECT", "default"))
    print(f"LangSmith tracing ON \u2014 project: {os.environ['LANGCHAIN_PROJECT']}")

# Langfuse: invoke/stream 호출 시 config={"callbacks": [langfuse_handler]} 전달
langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler
    langfuse_handler = CallbackHandler()
    print(f"Langfuse tracing ON \u2014 {os.environ.get('LANGFUSE_HOST', '')}")

# Langfuse config: pass to invoke/stream/batch calls
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}


LangSmith tracing ON — project: day1-labs-test


In [3]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)

# import 전에 강제로 환경변수 set
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = os.environ["LANGSMITH_API_KEY"]
os.environ["LANGCHAIN_PROJECT"] = os.environ.get("LANGSMITH_PROJECT", "day1-labs-test")

# 강제 flush 확인용
from langsmith import Client
client = Client()
print("LangSmith 연결 OK, 프로젝트:", os.environ["LANGCHAIN_PROJECT"])
print("API key tail:", os.environ["LANGCHAIN_API_KEY"][-6:])


LangSmith 연결 OK, 프로젝트: day1-labs-test
API key tail: a9d7da


## 7.2 검색 도구 정의

Tavily 클라이언트를 래핑하는 검색 함수를 만듭니다.
**docstring**과 **타입 힌트**가 에이전트에 도구 스키마를 알려줍니다.

**도구 함수 작성 규칙:**

`create_deep_agent()`의 `tools` 파라미터에 전달할 검색 함수를 정의합니다. Deep Agents는 함수의 docstring을 도구 설명으로, 타입 힌트를 파라미터 스키마로 자동 변환합니다. 따라서:

- **docstring**: 에이전트가 "이 도구를 언제 사용해야 하는지" 판단하는 근거가 됩니다. 명확하고 구체적으로 작성하세요.
- **타입 힌트**: 에이전트가 올바른 타입의 인자를 전달하도록 합니다. `Literal` 타입을 사용하면 허용 값을 제한할 수 있습니다.
- **Args 섹션**: 각 파라미터의 용도를 설명하면 에이전트가 더 정확하게 인자를 선택합니다.

In [4]:
# import sys
# !{sys.executable} -m pip install tavily-python

In [5]:
from typing import Literal
from tavily import TavilyClient

tavily = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])

def internet_search(
    query: str,
    max_results: int = 3,
    topic: Literal["general", "news"] = "general",
) -> dict:
    """인터넷에서 정보를 검색합니다.

    Args:
        query: 검색 쿼리
        max_results: 최대 결과 수
        topic: 검색 주제 카테고리
    """
    return tavily.search(query, max_results=max_results, topic=topic)

print("\u2713 검색 도구 준비 완료")

✓ 검색 도구 준비 완료


## 7.3 Deep Agents 리서치 에이전트

`create_deep_agent()`에 검색 도구와 시스템 프롬프트를 전달합니다.

**에이전트의 자동 워크플로:**

에이전트는 사용자의 요청을 받으면 다음과 같은 과정을 자동으로 수행합니다:

1. **계획 수립**: 빌트인 `write_todos` 도구로 작업을 단계별로 분해합니다.
2. **리서치 수행**: 전달된 검색 도구(`internet_search`)를 사용하여 웹에서 정보를 수집합니다.
3. **컨텍스트 관리**: 필요 시 파일 시스템 도구(`write_file`, `read_file`)로 중간 결과를 저장하여 토큰 한도를 관리합니다.
4. **결과 종합**: 수집한 정보를 분석하고 일관된 보고서로 종합합니다.

복잡한 작업의 경우, 에이전트는 전문 서브에이전트를 생성하여 특정 하위 작업의 컨텍스트를 격리할 수도 있습니다.

In [6]:
from deepagents import create_deep_agent

research_agent = create_deep_agent(
    model=model,
    tools=[internet_search],
    system_prompt="당신은 전문 리서처입니다. 웹을 검색한 후 결과를 한국어로 요약하세요.",
)
print("\u2713 리서치 에이전트 생성 완료")

✓ 리서치 에이전트 생성 완료


In [7]:
result = research_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "LangGraph가 무엇인지 검색해서 3줄로 요약해 주세요."
            }
        ]
    },
    config=lf_config,
)

print(result["messages"][-1].content)

LangGraph는 LangChain 생태계의 오픈소스 프레임워크로, LLM 에이전트와 워크플로를 그래프 구조로 설계·오케스트레이션할 수 있게 해줍니다.  
각 단계를 노드와 엣지로 표현해 상태 관리, 분기, 반복, 멀티에이전트 흐름을 더 정밀하게 제어할 수 있습니다.  
특히 장기 실행, 실패 후 재개, 스트리밍, human-in-the-loop 같은 신뢰성 높은 에이전트 시스템 구축에 강점이 있습니다.


## 7.4 스트리밍으로 과정 관찰

`stream(mode="updates")`로 에이전트가 어떤 단계를 거치는지 실시간으로 확인합니다.

**LangGraph 스트리밍 시스템:**

LangGraph는 완전한 응답이 준비되기 전에 진행 상황을 점진적으로 표시하여 애플리케이션의 반응성을 높이는 포괄적인 스트리밍 시스템을 제공합니다.

| 스트림 모드 | 용도 |
|---|---|
| `values` | 각 그래프 단계 후 **전체 상태**를 스트리밍 |
| `updates` | 각 단계 후 **상태 변경분만** 스트리밍 |
| `messages` | LLM 토큰을 메타데이터와 함께 스트리밍 |
| `custom` | 노드에서 사용자 정의 데이터를 스트리밍 |
| `debug` | 포괄적인 실행 정보를 스트리밍 |

`stream()` (동기) 또는 `astream()` (비동기) 메서드로 스트리밍에 접근하며, 여러 모드를 리스트로 전달하여 동시에 사용할 수도 있습니다. 아래 예제에서는 `updates` 모드를 사용하여 에이전트의 각 단계(도구 호출, 최종 응답)를 실시간으로 출력합니다.

In [8]:
for chunk in research_agent.stream(
    {
        "messages": [
            {
                "role": "user",
                "content": "LangChain v1의 주요 변경사항을 검색해서 요약해 주세요."
            }
        ]
    },
    stream_mode="updates",
    config=lf_config,
):
    for node_name, node_data in chunk.items():
        if not node_data:
            continue

        msgs = node_data.get("messages", [])

        if hasattr(msgs, "value"):
            msgs = msgs.value

        if not msgs:
            continue

        last = msgs[-1]

        if hasattr(last, "tool_calls") and last.tool_calls:
            for tc in last.tool_calls:
                print(
                    f"[도구 호출] {tc['name']}({tc['args'].get('query', '')[:50]})"
                )

        elif hasattr(last, "content") and last.content and not hasattr(last, "tool_call_id"):
            content = last.content if isinstance(last.content, str) else str(last.content)

            if content.strip():
                print(f"\n[최종 응답]\n{content}")


[최종 응답]
LangChain v1의 주요 변경사항을 검색해서 요약해 주세요.
[도구 호출] internet_search(LangChain v1 주요 변경사항 release notes migration guide)
[도구 호출] internet_search(LangChain v1 release notes breaking changes migrat)
[도구 호출] internet_search(site:docs.langchain.com/oss/python/releases/langch)
[도구 호출] internet_search(site:docs.langchain.com/oss/python/migrate/langcha)

[최종 응답]
LangChain v1의 주요 변경사항을 요약하면 다음과 같습니다.

### 핵심 변화
- **`create_agent` 중심으로 에이전트 API 단순화**
  - 기존의 복잡한 에이전트 구성보다 더 단순한 진입점 제공
  - 내부적으로는 **LangGraph 기반**으로 동작해서 확장성과 제어력이 높아짐

- **미들웨어(middleware) 도입**
  - v1에서 가장 큰 구조적 변화 중 하나
  - 모델 호출 전/후 처리, 동적 모델 선택, 툴 호출 제어, 로깅, 가드레일 같은 로직을 미들웨어로 구성 가능
  - 예전의 hook/pre/post model 방식이 미들웨어 패턴으로 정리됨

- **구조화된 출력(Structured Output) 방식 개편**
  - 출력 스키마를 더 일관된 방식으로 다룰 수 있게 변경
  - 일부 예전 방식은 제거되거나 권장되지 않고, `toolStrategy`/`providerStrategy` 같은 새 전략 기반 접근으로 이동

- **표준 콘텐츠 블록(Standard Content Blocks) 도입**
  - 멀티모달/메시지 콘텐츠 표현 방식이 더 표준화됨
  - 텍스트, 이미지 등 다양한 콘텐츠를 일관된 구조로 처리 가능

- **패키지/네임스페이스 정리**
  - 패키지 구조가 더 

## 7.5 LangChain 에이전트로 비교

같은 검색 도구를 LangChain `create_agent()`로도 사용해 봅니다.

**LangChain 에이전트와의 차이점:**

LangChain의 `create_agent()`는 모델과 도구를 받아 간단한 ReAct 에이전트를 생성합니다. Deep Agents와 비교하면:

- **LangChain**: 도구 호출 에이전트의 기본 형태. 빠른 프로토타이핑에 적합하지만, 태스크 플래닝이나 파일 시스템 관리 같은 고급 기능은 직접 구현해야 합니다.
- **Deep Agents**: 플래닝(`write_todos`), 파일 관리, 서브에이전트 위임이 기본 내장되어 있어, 복잡한 멀티스텝 작업에 더 적합합니다.

`@tool` 데코레이터를 사용하면 LangChain의 도구 인터페이스에 맞게 함수를 변환할 수 있습니다. 시스템 프롬프트와 도구 리스트를 `create_agent()`에 전달하는 패턴은 Deep Agents와 동일합니다.

In [9]:
from langchain.agents import create_agent
from langchain.tools import tool

@tool
def search_web(query: str) -> dict:
    """웹에서 정보를 검색합니다."""
    return tavily.search(query, max_results=3)

lc_agent = create_agent(
    model=model,
    tools=[search_web],
    system_prompt="당신은 리서치 어시스턴트입니다. 한국어로 답변하세요.",
)

result = lc_agent.invoke(
    {"messages": [{"role": "user", "content": "LangChain v1의 주요 특징을 검색해서 알려주세요."}]},
    config=lf_config,
)
print(result["messages"][-1].content)

검색 결과를 바탕으로 정리하면, **LangChain v1의 주요 특징**은 다음과 같습니다.

## 1) 더 단순해진 에이전트 API: `create_agent`
LangChain v1은 에이전트를 만드는 기본 진입점을 **`create_agent`** 중심으로 단순화했습니다.  
기존보다 더 쉽게 에이전트를 만들 수 있으면서도, 내부적으로는 확장성이 높아졌습니다.

- 모델 호출
- 툴 선택 및 실행
- 더 이상 툴이 필요 없으면 종료

같은 기본 에이전트 루프를 표준화해 제공합니다.

## 2) 미들웨어 기반 커스터마이징
v1의 핵심 변화 중 하나는 **미들웨어(Middleware)** 입니다.  
개발자는 다음 지점에 로직을 끼워 넣을 수 있습니다.

- 모델 호출 전/후
- 툴 호출 전/후
- 컨텍스트 주입
- 가드레일, 로깅, 정책 적용

즉, 단순한 체인 조합을 넘어서 **운영 환경에 맞는 에이전트 제어**가 쉬워졌습니다.

## 3) LangGraph 기반 아키텍처
LangChain v1은 **LangGraph 위에 구축**되어 있습니다.  
이 덕분에 다음과 같은 장점이 있습니다.

- 더 안정적인 에이전트 실행
- 상태 관리 향상
- 복잡한 워크플로우 구성 용이
- 프로덕션 환경에 적합한 구조

즉, “간단한 API”는 유지하면서도, 내부적으로는 더 견고한 실행 엔진을 사용합니다.

## 4) 구조화된 출력(Structured Output) 지원 강화
v1은 모델 출력 결과를 더 일관되게 다루기 위해 **구조화된 출력**을 강화했습니다.

- JSON 스키마 기반 출력
- 예측 가능한 응답 형식
- 후처리 및 시스템 연동 용이

이 기능은 특히 API 응답 생성, 데이터 추출, 자동화 파이프라인에서 유용합니다.

## 5) 표준 콘텐츠 블록(Standard Content Blocks)
모델마다 멀티모달 입력/출력 방식이 조금씩 다른데, v1은 이를 **표준 콘텐츠 블록** 형태로 정리하려고 합니다.

- 텍스트
- 이미지
- 기타 멀티모달 요

## 요약

이 미니 프로젝트에서 사용한 기술:

| 기술 | 출처 |
|---|---|
| `ChatOpenAI` + `load_dotenv` | 00_setup |
| 메시지 역할, 스트리밍 | 01_llm_basics |
| `@tool`, `create_agent()` | 02_langchain_basics |
| `InMemorySaver`, `thread_id` | 03_langchain_memory |
| `StateGraph`, `compile()` | 04_langgraph_basics |
| `create_deep_agent()` | 05_deep_agents_basics |

### 다음 단계
→ 중급 과정으로 진행하세요! **[06_comparison.ipynb](./06_comparison.ipynb)** 에서 안내를 확인하세요.
